# 1. Imports y config

In [1]:
import numpy as np
import pandas as pd
import os

In [2]:
from config import CONFIG

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
np.random.seed(42)

# 2. Carga de datos

In [4]:
from models import load_data

In [5]:
spain_df, spain_feats = load_data(CONFIG["out_features_spain"], CONFIG["reg_path_spain"])

Datos cargados desde data/processed/features_df_spain.parquet: (799338, 132)
    Número de estaciones de datos: 66
Registro de características cargado desde data/processed/feature_reg_spain.csv
    Longitud vista completa: 100
    Longitud vista mínima: 28


In [6]:
clm_df, clm_feats = load_data(CONFIG["out_features_clm"], CONFIG["reg_path_clm"])

Datos cargados desde data/processed/features_df_clm.parquet: (339698, 130)
    Número de estaciones de datos: 62
Registro de características cargado desde data/processed/feature_reg_clm.csv
    Longitud vista completa: 100
    Longitud vista mínima: 28


# 3. Pruebas

In [7]:
from models import feature_view, keep_mask, load_experiment

## 3.1. Vistas

In [8]:
print(f"Primeras 8 columnas de lista completa ({len(feature_view(spain_feats, 'curated'))}):")
print(f"    {feature_view(spain_feats, 'curated')[:8]}")
print(f"Primeras 8 columnas de vista minima ({len(feature_view(spain_feats, 'minimal'))}):")
print(f"    {feature_view(spain_feats, 'minimal')[:8]}")

Primeras 8 columnas de lista completa (100):
    ['altitud', 'anio', 'cos_doy', 'days_since_prec', 'days_since_prec_ex', 'days_since_racha_ex', 'days_since_tmax_ex', 'days_since_tmin_ex']
Primeras 8 columnas de vista minima (28):
    ['altitud', 'anio', 'cos_doy', 'fecha', 'hrMedia', 'indicativo', 'koppen_class', 'koppen_main']


## 3.2. Eliminación de registros que se asoman a splits posteriores 

In [9]:
keep = keep_mask(spain_df, 7)
print(f"Se guarda un {keep.mean():.3%} de los registros")
print("Últimos 8 registros eliminados del conjunto de entrenamiento para N=7:")
for _, row in spain_df.loc[~keep].loc[(spain_df["split"] == "train")].tail(8).iterrows():
    print(f"    {row.nombre} - {row.fecha.date()}")

Se guarda un 99.827% de los registros
Últimos 8 registros eliminados del conjunto de entrenamiento para N=7:
    GRAN CANARIA AEROPUERTO - 2018-12-31
    HIERRO AEROPUERTO - 2018-12-25
    HIERRO AEROPUERTO - 2018-12-26
    HIERRO AEROPUERTO - 2018-12-27
    HIERRO AEROPUERTO - 2018-12-28
    HIERRO AEROPUERTO - 2018-12-29
    HIERRO AEROPUERTO - 2018-12-30
    HIERRO AEROPUERTO - 2018-12-31


## 3.3. Creación de experimento

In [10]:
_ex = load_experiment(spain_df, "tmax", 1, "curated", spain_feats)
print(f"Experimento para tmax con N={_ex["N"]}")
print(f"    primeras 5 columnas {_ex["cols"][:5]}")
for split in ["train", "val", "test"]:
    print(f"    {split}:\tX={_ex[split]["X"].shape}\tprevalencia={_ex[split]["y"].mean():.2%}")

Experimento para tmax con N=1
    primeras 5 columnas ['altitud', 'anio', 'cos_doy', 'days_since_prec', 'days_since_prec_ex']
    train:	X=(654916, 100)	prevalencia=5.19%
    val:	X=(72186, 100)	prevalencia=8.12%
    test:	X=(72038, 100)	prevalencia=14.03%


## 3.4. Prueba de baseline de persistencia

In [11]:
from models import conditional_probability_baseline

In [12]:
_base = conditional_probability_baseline(spain_df, "tmax", 1, spain_feats)
print("Baseline de persistencia para tmax con N=1 calculado sobre train")
unique_probs = np.unique(_base["train"])
print(f"   P(target=1 | extremo=1): {unique_probs.max()}")
print(f"   P(target=1 | extremo=0): {unique_probs.min()}")

Baseline de persistencia para tmax con N=1 calculado sobre train
   P(target=1 | extremo=1): 0.3497529702387954
   P(target=1 | extremo=0): 0.03563145824206973


## 3.5. Prueba de métricas con baseline de probabilidad condicional

In [13]:
from models import evaluate

In [14]:
res = evaluate(_ex["train"]["y"], _base["train"])
print("Resultados de evaluación de probabilidad condicional para tmax con N=1:")
for metric, value in res.items():
    if metric != "confusion":
        print(f"{metric:<9}: {value:0.4f}")
    else:
        print(f"{metric:<9}:\n{pd.DataFrame(value)}")

Resultados de evaluación de probabilidad condicional para tmax con N=1:
prevalence: 0.0519
AP       : 0.1561
ROC_AUC  : 0.6570
threshold: 0.0400
F1       : 0.3497
precision: 0.3498
recall   : 0.3496
accuracy : 0.9325
confusion:
        0      1
0  598788  22111
1   22124  11893


## 3.6. Prueba del preprocesado

In [15]:
from models import GroupedTimeWindowStandardScaler, GroupedMinMaxScaler

In [16]:
GroupedTimeWindowStandardScaler(
    "indicativo", "fecha", CONFIG["core_half_window"], CONFIG["core_scale_cols"]
).fit_transform(_ex["train"]["X"])[["tmax", "tmin", "prec", "racha"]].head()

,tmax,tmin,prec,racha
0,-0.768698,0.223038,-0.282523,-0.920079
1,-0.046475,-0.035443,-0.281797,-1.074788
2,-0.409557,0.808258,-0.159574,-0.586298
3,0.109955,0.979524,0.034640,-0.233185
4,-0.263630,0.094453,-0.302371,-0.555037


In [17]:
GroupedMinMaxScaler(
    "indicativo", ["tmax_diff_1"], feature_range=(-1, 1)
).fit_transform(_ex["train"]["X"])[["tmax", "tmax_diff_1"]].head(10)

,tmax,tmax_diff_1
0,12.6,NaN
1,14.7,0.358779
2,13.4,0.099237
3,15.0,0.320611
4,13.6,0.091603
5,14.6,0.274809
6,10.5,-0.114504
7,13.0,0.389313
8,11.9,0.114504
9,12.6,0.251909


In [18]:
from models import make_preprocessor

In [19]:
pre = make_preprocessor(_ex["train"]["X"], _ex["cols"], passthrough=False, core_only=True)
pre

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('impute_encode', ...), ('core_scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('impute', ...), ('encode', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"verbose_feature_names_out verbose_

In [20]:
pre_train = pre.fit_transform(_ex["train"]["X"]).head()
print(f"{pre_train.isna().sum().sum()} valores perdidos en train")
pre_train.head()

0 valores perdidos en train


,altitud,anio,cos_doy,days_since_prec,days_since_prec_ex,days_since_racha_ex,days_since_tmax_ex,days_since_tmin_ex,extreme_prec_lag_1,extreme_racha_lag_1,extreme_tmax_lag_1,extreme_tmin_lag_1,hrMedia,latitud,longitud,mslp_max,mslp_max_diff_1,mslp_max_diff_3,mslp_max_lag_1,mslp_max_lag_3,mslp_media,mslp_min,mslp_min_diff_1,mslp_min_diff_3,mslp_min_lag_1,mslp_min_lag_3,prec,prec_diff_1,prec_diff_3,prec_dist_to_ex,prec_ewma_14,prec_ewma_3,prec_ewma_7,prec_ex_today,prec_lag_1,prec_lag_3,prec_roll_14_max,prec_roll_14_mean,prec_roll_14_sum,prec_roll_3_max,prec_roll_3_mean,prec_roll_3_sum,prec_roll_7_max,prec_roll_7_mean,prec_roll_7_sum,racha,racha_diff_1,racha_diff_3,racha_dist_to_ex,racha_ewma_14,racha_ewma_3,racha_ewma_7,racha_ex_today,racha_lag_1,racha_lag_3,racha_roll_14_max,racha_roll_14_mean,racha_roll_3_max,racha_roll_3_mean,racha_roll_7_max,racha_roll_7_mean,sin_doy,tmax,tmax_diff_1,tmax_diff_3,tmax_dist_to_ex,tmax_ewma_14,tmax_ewma_3,tmax_ewma_7,tmax_ex_today,tmax_lag_1,tmax_lag_3,tmax_roll_14_max,tmax_roll_14_mean,tmax_roll_3_max,tmax_roll_3_mean,tmax_roll_7_max,tmax_roll_7_mean,tmed,tmin,tmin_diff_1,tmin_diff_3,tmin_dist_to_ex,tmin_ewma_14,tmin_ewma_3,tmin_ewma_7,tmin_ex_today,tmin_lag_1,tmin_lag_3,tmin_roll_14_max,tmin_roll_14_mean,tmin_roll_3_max,tmin_roll_3_mean,tmin_roll_7_max,tmin_roll_7_mean,velmedia,koppen_main_B,koppen_main_C,koppen_class_BSh,koppen_class_BSk,koppen_class_BWh,koppen_class_Cfa,koppen_class_Cfb,koppen_class_Csa,koppen_class_Csb
0,71.0,1990.0,0.999852,3.0,18.0,17.0,22.0,23.0,0.0,0.0,0.0,0.0,1.495262,41.145,1.163611,-0.176379,-0.033081,-0.041809,1019.331238,1019.329926,-0.040223,0.077678,-0.024353,-0.057312,1015.371643,1015.369995,-0.276154,0.0,0.0,-7.01,0.000000,0.000000,0.000000,0.0,0.0,0.0,3.7,0.392857,5.5,0.0,0.000000,0.0,0.8,0.128571,0.9,-0.932126,0.0,0.0,-16.725000,5.000000,5.000000,5.000000,0.0,9.7,9.7,15.8,10.042857,11.9,9.833333,13.9,9.985714,0.017202,-0.768698,0.100000,0.000000,-7.580000,12.600000,12.600000,12.600000,0.0,21.6,21.6,25.799999,21.628572,23.1,21.633333,24.5,21.642857,-0.214955,0.223038,-0.1,0.0,-7.40,5.400000,5.400000,5.400000,0.0,12.3,12.3,15.6,12.085714,13.6,12.200000,14.8,12.142857,-1.112509,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,71.0,1990.0,0.999408,3.0,18.0,17.0,22.0,23.0,0.0,0.0,0.0,0.0,1.458801,41.145,1.163611,-0.290058,-0.920227,-0.041809,1021.398560,1019.329926,-0.161453,-0.048223,-1.525269,-0.057312,1017.868469,1015.369995,-0.273308,0.0,0.0,-7.02,0.000000,0.000000,0.000000,0.0,0.0,0.0,3.7,0.392857,5.5,0.0,0.000000,0.0,0.8,0.128571,0.9,-1.088441,-0.8,0.0,-18.014999,4.571428,4.466667,4.542857,0.0,5.0,9.7,15.8,10.042857,11.9,9.833333,13.9,9.985714,0.034398,-0.061022,2.099999,0.000000,-5.485002,13.725000,14.000000,13.800000,0.0,12.6,21.6,25.799999,21.628572,23.1,21.633333,24.5,21.642857,-0.075500,-0.047559,-1.2,0.0,-6.20,4.757143,4.600000,4.714286,0.0,5.4,12.3,15.6,12.085714,13.6,12.200000,14.8,12.142857,-1.126200,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,71.0,1990.0,0.998669,0.0,18.0,17.0,22.0,23.0,0.0,0.0,0.0,0.0,1.517506,41.145,1.163611,-0.809166,-4.168152,-0.041809,1020.478333,1019.329926,-0.682369,-0.558287,-4.874023,-0.057312,1016.343201,1015.369995,-0.145147,0.5,0.0,-5.75,0.191002,0.285714,0.216216,0.0,0.0,0.0,3.7,0.392857,5.5,0.5,0.166667,0.5,0.8,0.128571,0.9,-0.593038,2.7,0.0,-15.300001,5.460951,5.857143,5.562162,0.0,4.2,9.7,15.8,10.042857,6.9,5.366667,13.9,9.985714,0.051584,-0.428876,-1.300000,0.000000,-6.790001,13.600849,13.657143,13.627027,0.0,14.7,21.6,25.799999,21.628572,14.7,13.566667,24.5,21.642857,0.285019,0.766837,3.3,0.0,-9.48,5.804924,6.257143,5.918919,0.0,4.2,12.3,15.6,12.085714,7.5,5.700000,14.8,12.142857,0.127882,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,71.0,1990.0,0.997634,0.0,18.0,17.0,22.0,23.0,0.0,0.0,0.0,0.0,0.443588,41.145,1.163611,0.384255,9.450073,4.361694,1016.310181,1021.398560,-0.000520,-0.327546,1.886414,-4.512878,1011.469177,1017.868469,0.054627,0.8,1.3,-6.05,0.530276,0.826667,0.612571,0.0,0.5,0.0,3.7,0.392857,5.5,1.3,0.600000,1.8,0.8,0.128571,0

In [21]:
pre_val = pre.transform(_ex["val"]["X"]).head()
print(f"{pre_val.isna().sum().sum()} valores perdidos en val")
pre_val.head()

0 valores perdidos en val


,altitud,anio,cos_doy,days_since_prec,days_since_prec_ex,days_since_racha_ex,days_since_tmax_ex,days_since_tmin_ex,extreme_prec_lag_1,extreme_racha_lag_1,extreme_tmax_lag_1,extreme_tmin_lag_1,hrMedia,latitud,longitud,mslp_max,mslp_max_diff_1,mslp_max_diff_3,mslp_max_lag_1,mslp_max_lag_3,mslp_media,mslp_min,mslp_min_diff_1,mslp_min_diff_3,mslp_min_lag_1,mslp_min_lag_3,prec,prec_diff_1,prec_diff_3,prec_dist_to_ex,prec_ewma_14,prec_ewma_3,prec_ewma_7,prec_ex_today,prec_lag_1,prec_lag_3,prec_roll_14_max,prec_roll_14_mean,prec_roll_14_sum,prec_roll_3_max,prec_roll_3_mean,prec_roll_3_sum,prec_roll_7_max,prec_roll_7_mean,prec_roll_7_sum,racha,racha_diff_1,racha_diff_3,racha_dist_to_ex,racha_ewma_14,racha_ewma_3,racha_ewma_7,racha_ex_today,racha_lag_1,racha_lag_3,racha_roll_14_max,racha_roll_14_mean,racha_roll_3_max,racha_roll_3_mean,racha_roll_7_max,racha_roll_7_mean,sin_doy,tmax,tmax_diff_1,tmax_diff_3,tmax_dist_to_ex,tmax_ewma_14,tmax_ewma_3,tmax_ewma_7,tmax_ex_today,tmax_lag_1,tmax_lag_3,tmax_roll_14_max,tmax_roll_14_mean,tmax_roll_3_max,tmax_roll_3_mean,tmax_roll_7_max,tmax_roll_7_mean,tmed,tmin,tmin_diff_1,tmin_diff_3,tmin_dist_to_ex,tmin_ewma_14,tmin_ewma_3,tmin_ewma_7,tmin_ex_today,tmin_lag_1,tmin_lag_3,tmin_roll_14_max,tmin_roll_14_mean,tmin_roll_3_max,tmin_roll_3_mean,tmin_roll_7_max,tmin_roll_7_mean,velmedia,koppen_main_B,koppen_main_C,koppen_class_BSh,koppen_class_BSk,koppen_class_BWh,koppen_class_Cfa,koppen_class_Cfb,koppen_class_Csa,koppen_class_Csb
0,71.0,2019.0,0.999852,8.0,42.0,62.0,28.0,63.0,0.0,0.0,0.0,0.0,-0.152387,41.145,1.163611,0.881797,0.407715,0.148438,1029.761841,1030.021118,0.991417,1.059252,0.306763,-0.154053,1027.038696,1027.499512,-0.276154,0.0,0.0,-7.01,0.061479,4.682292e-06,0.005420,0.0,0.0,0.0,3.7,0.392857,5.5,0.0,0.0,0.0,0.0,0.0,0.0,-0.446992,3.1,0.6,-13.925000,7.091407,6.423162,6.287932,0.0,4.7,7.2,17.5,7.421429,7.8,5.566667,7.8,5.914286,0.017202,1.037162,-0.799999,4.000000,-2.180000,16.475911,17.477973,16.671462,0.0,18.799999,14.000000,19.4,16.714286,18.799999,17.633333,18.799999,15.585714,-0.087575,-0.856934,0.6,-0.3,-2.80,2.785871,0.727884,1.621254,0.0,0.2,1.1,6.9,3.342857,0.8,0.333333,5.6,1.957143,-0.638286,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,71.0,2019.0,0.999408,9.0,43.0,63.0,29.0,64.0,0.0,0.0,0.0,0.0,0.418467,41.145,1.163611,0.569148,-2.754028,-2.695435,1030.169556,1030.110962,0.719572,0.826537,-2.653320,-1.989502,1027.345459,1026.681641,-0.273308,0.0,0.0,-7.02,0.052889,2.338855e-06,0.004037,0.0,0.0,0.0,3.7,0.392857,5.5,0.0,0.0,0.0,0.0,0.0,0.0,-0.814068,-2.0,1.6,-16.415001,6.919220,6.111581,6.165949,0.0,7.8,4.2,10.3,6.585714,7.8,6.100000,7.8,6.028571,0.034398,0.003589,-3.100000,-1.200001,-5.285002,16.265790,16.188986,16.228597,0.0,18.000000,16.100000,19.4,16.607143,18.799999,17.233333,18.799999,15.500000,-0.741750,-1.080452,-1.2,-0.4,-1.60,2.361088,0.163942,1.115941,0.0,0.8,0.0,6.1,2.821429,0.8,0.200000,3.6,1.100000,-0.638262,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,71.0,2019.0,0.998669,10.0,44.0,64.0,30.0,0.0,0.0,0.0,0.0,0.0,0.201880,41.145,1.163611,0.972090,3.289185,0.942871,1027.415527,1029.761841,1.049967,1.086816,2.683838,0.337280,1024.692139,1027.038696,-0.290095,0.0,0.0,-6.25,0.045546,1.168856e-06,0.003013,0.0,0.0,0.0,3.7,0.392857,5.5,0.0,0.0,0.0,0.0,0.0,0.0,-0.628194,0.9,2.0,-15.500001,6.889990,6.405790,6.299462,0.0,5.8,4.7,10.3,6.328571,7.8,6.766667,7.8,6.028571,0.051584,-0.428876,-1.500000,-5.400000,-6.790001,15.883684,14.794493,15.521447,0.0,14.900000,18.799999,19.4,16.392857,18.000000,15.433333,18.799999,15.542857,-1.269824,-1.568625,-2.1,-2.7,0.52,1.712943,-1.168029,0.211955,1.0,-0.4,0.2,6.1,2.328571,0.8,-0.700000,2.4,0.228571,-0.393785,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,71.0,2019.0,0.997634,11.0,45.0,65.0,31.0,0.0,0.0,0.0,0.0,1.0,-0.589669,41.145,1.163611,1.204464,2.215576,2.750732,1030.704712,1030.169556,1.144253,1.066293,0.299072,0.329590,1027.375977,1027.345459,-0.286338,0.0,0.0,-7.35,0.039258,5.842850e-07,0.002251,0.0,0.0,0.0,3.7,0.392857,5.5,0.0,0.0,0.0,0.0,0.0,0.0,-0.94882

## 3.7. Prueba de experimento

In [22]:
from models import run_experiment

In [23]:
_, _ = run_experiment(spain_df, "tmax", 1, "Baseline", "curated", spain_feats, "spain")

[tmax, N=1, Baseline]
    Métricas en train
        AP=0.156 | ROC_AUC=0.657
        prev=0.052 | P=0.350 | R=0.350 | F1=0.350 | acc=0.932
    Métricas en val
        AP=0.231 | ROC_AUC=0.689
        prev=0.081 | P=0.430 | R=0.428 | F1=0.429 | acc=0.908
    Métricas en test
        AP=0.344 | ROC_AUC=0.725
        prev=0.140 | P=0.526 | R=0.528 | F1=0.527 | acc=0.867


# 4. Entrenamiento

In [24]:
from models import run_all_experiments

## 4.1. España

### 4.1.1. Vista completa

In [25]:
res_spain_full = run_all_experiments(spain_df, "curated", spain_feats, "spain", verbose=True)

Ejecutando 60 experimentos (4 vars x 3 N x 5 modelos)
Número de iteraciones por modelo: {'NaiveBayes': 10, 'LogReg': 10, 'DecisionTree': 10, 'LGBM': 10} | jobs paralelos: 1

[ 1/60] tmax N=1 Baseline
[tmax, N=1, Baseline]
    Métricas en train
        AP=0.156 | ROC_AUC=0.657
        prev=0.052 | P=0.350 | R=0.350 | F1=0.350 | acc=0.932
    Métricas en val
        AP=0.231 | ROC_AUC=0.689
        prev=0.081 | P=0.430 | R=0.428 | F1=0.429 | acc=0.908
    Métricas en test
        AP=0.344 | ROC_AUC=0.725
        prev=0.140 | P=0.526 | R=0.528 | F1=0.527 | acc=0.867
    Hecho en 1s | transcurrido 0.0 min | tiempo restante estimado ~1.3 min (59 modelos restantes)

[ 2/60] tmax N=1 NaiveBayes
Fitting 1 folds for each of 10 candidates, totalling 10 fits
[CV 1/1] END classifier__var_smoothing=9.91564456663839e-07;, score=0.295 total time=   1.0s
[CV 1/1] END classifier__var_smoothing=0.04033800832600378;, score=0.118 total time=   0.6s
[CV 1/1] END classifier__var_smoothing=0.0007177141927992

### 4.1.2. Vista mínima

In [26]:
res_spain_min = run_all_experiments(spain_df, "minimal", spain_feats, "spain", verbose=True)

Ejecutando 60 experimentos (4 vars x 3 N x 5 modelos)
Número de iteraciones por modelo: {'NaiveBayes': 10, 'LogReg': 10, 'DecisionTree': 10, 'LGBM': 10} | jobs paralelos: 1

[ 1/60] tmax N=1 Baseline
[tmax, N=1, Baseline]
    Métricas en train
        AP=0.156 | ROC_AUC=0.657
        prev=0.052 | P=0.350 | R=0.350 | F1=0.350 | acc=0.932
    Métricas en val
        AP=0.231 | ROC_AUC=0.689
        prev=0.081 | P=0.430 | R=0.428 | F1=0.429 | acc=0.908
    Métricas en test
        AP=0.344 | ROC_AUC=0.725
        prev=0.140 | P=0.526 | R=0.528 | F1=0.527 | acc=0.867
    Hecho en 1s | transcurrido 0.0 min | tiempo restante estimado ~1.4 min (59 modelos restantes)

[ 2/60] tmax N=1 NaiveBayes
Fitting 1 folds for each of 10 candidates, totalling 10 fits
[CV 1/1] END classifier__var_smoothing=9.91564456663839e-07;, score=0.384 total time=   0.2s
[CV 1/1] END classifier__var_smoothing=0.04033800832600378;, score=0.260 total time=   0.2s
[CV 1/1] END classifier__var_smoothing=0.0007177141927992

## 4.2. CLM

### 4.2.1. Vista completa

In [27]:
res_clm_full = run_all_experiments(clm_df, "curated", clm_feats, "clm", verbose=True)

Ejecutando 60 experimentos (4 vars x 3 N x 5 modelos)
Número de iteraciones por modelo: {'NaiveBayes': 10, 'LogReg': 10, 'DecisionTree': 10, 'LGBM': 10} | jobs paralelos: 1

[ 1/60] tmax N=1 Baseline
[tmax, N=1, Baseline]
    Métricas en train
        AP=0.167 | ROC_AUC=0.658
        prev=0.060 | P=0.358 | R=0.357 | F1=0.358 | acc=0.923
    Métricas en val
        AP=0.361 | ROC_AUC=0.755
        prev=0.103 | P=0.563 | R=0.561 | F1=0.562 | acc=0.909
    Métricas en test
        AP=0.456 | ROC_AUC=0.780
        prev=0.162 | P=0.628 | R=0.631 | F1=0.630 | acc=0.880
    Hecho en 1s | transcurrido 0.0 min | tiempo restante estimado ~0.9 min (59 modelos restantes)

[ 2/60] tmax N=1 NaiveBayes
Fitting 1 folds for each of 10 candidates, totalling 10 fits
[CV 1/1] END classifier__var_smoothing=9.91564456663839e-07;, score=0.400 total time=   0.2s
[CV 1/1] END classifier__var_smoothing=0.04033800832600378;, score=0.200 total time=   0.2s
[CV 1/1] END classifier__var_smoothing=0.0007177141927992

### 4.2.2. Vista mínima

In [28]:
res_clm_min = run_all_experiments(clm_df, "minimal", clm_feats, "clm", verbose=True)

Ejecutando 60 experimentos (4 vars x 3 N x 5 modelos)
Número de iteraciones por modelo: {'NaiveBayes': 10, 'LogReg': 10, 'DecisionTree': 10, 'LGBM': 10} | jobs paralelos: 1

[ 1/60] tmax N=1 Baseline
[tmax, N=1, Baseline]
    Métricas en train
        AP=0.167 | ROC_AUC=0.658
        prev=0.060 | P=0.358 | R=0.357 | F1=0.358 | acc=0.923
    Métricas en val
        AP=0.361 | ROC_AUC=0.755
        prev=0.103 | P=0.563 | R=0.561 | F1=0.562 | acc=0.909
    Métricas en test
        AP=0.456 | ROC_AUC=0.780
        prev=0.162 | P=0.628 | R=0.631 | F1=0.630 | acc=0.880
    Hecho en 1s | transcurrido 0.0 min | tiempo restante estimado ~0.8 min (59 modelos restantes)

[ 2/60] tmax N=1 NaiveBayes
Fitting 1 folds for each of 10 candidates, totalling 10 fits
[CV 1/1] END classifier__var_smoothing=9.91564456663839e-07;, score=0.533 total time=   0.1s
[CV 1/1] END classifier__var_smoothing=0.04033800832600378;, score=0.319 total time=   0.1s
[CV 1/1] END classifier__var_smoothing=0.0007177141927992